In [ ]:
def operator_selection_prompt_from_history(
    operator_summary,
    current_pair,
    iteration
):
    
    prompt = f"""
    You are a Large Language Model acting as an adaptive operator selector for a Differential Evolution algorithm.
    
    The algorithm has externally triggered an operator re-selection event.
    This event may be due to stagnation, weak improvement, or diversity loss.
    The numerical detection was already performed outside the LLM.
    
    Your task is not to detect stagnation.
    Your task is not to analyze population diversity.
    Your task is not to analyze candidate solutions.
    Your task is only to select the next mutation-crossover pair based on the summarized history of previously tested configurations.
    
    Assume this is a minimization problem.
    
    Available mutation strategies:
    1: "DE/rand/1"
    2: "DE/rand/2"
    3: "DE/current-to-rand/1"
    4: "Gaussian-based mutation"
    5: "DE/best/1"
    6: "Trigonometric Mutation"
    7: "Reflection-based Mutation"
    8: "DE/current-to-pbest/1"
    9: "Hemostasis-based Mutation"
    10: "UDE Mutation"
    
    Available crossover strategies:
    1: "Binomial Crossover"
    2: "Exponential Crossover"
    3: "Continuous Crossover"
    4: "Directional Crossover"
    5: "SBCX Crossover"
    
    Current iteration:
    {iteration}
    
    Current active pair:
    {current_pair}
    
    Summary of previously tested operator configurations:
    {operator_summary}
    
    How to interpret the summary:
    - times_used indicates how many times a pair has been applied.
    - total_improvement is the accumulated fitness improvement produced by that pair.
    - mean_improvement is the average improvement per use of that pair.
    - best_fitness_reached is the best fitness value obtained after using that pair.
    - In this minimization problem, larger positive improvement is better.
    - A pair with high mean_improvement but low times_used may be promising but has limited evidence.
    - A pair with high total_improvement but low mean_improvement may have worked because it was used many times, not necessarily because it is efficient per use.
    - A pair with high times_used and low mean_improvement may be less attractive.
    - A pair with zero total_improvement and zero mean_improvement should usually be avoided unless there is no better alternative.
    - A pair that is not present in the summary has not been tested yet.
    
    Exploration schedule:
    - During early iterations, give higher priority to exploration of new or rarely tested configurations.
    - For iterations 0 to 400, prefer untested or rarely tested mutation-crossover pairs whenever reasonable.
    - For iterations 0 to 400, do not over-prioritize total_improvement, because there is not enough historical evidence yet.
    - For iterations 0 to 400, avoid selecting a pair that has already been used many times unless all other alternatives are clearly poor.
    - For iterations 401 to 450, use a balanced strategy: combine historically successful pairs with less-tested alternatives.
    - For iterations greater than 450, give more importance to mean_improvement, best_fitness_reached, and reliable historical evidence.
    - Even in later iterations, avoid excessive reuse of a single pair if it has recently been flagged by the external numerical criterion.
    
    Selection rules:
    - Select exactly one mutation-crossover pair.
    - The current active pair was already flagged by the external numerical criterion.
    - Do not return the current active pair.
    - Do not select the same pair only because it has the largest total_improvement.
    - Strong historical performance is useful, but it must be balanced against times_used.
    - Avoid repeatedly selecting the most-used pair if other reasonable alternatives exist.
    - Prefer pairs with positive mean_improvement and moderate times_used.
    - Prefer pairs that achieved a good best_fitness_reached without being overused.
    - Give special attention to pairs with high mean_improvement, because they were effective per use.
    - Consider times_used when judging reliability: a pair used once may be promising, but the evidence is limited.
    - Avoid pairs that have repeatedly produced zero or very small improvement.
    - If the best historical pair has been used many times, prefer a different pair with similar behavior or a different crossover.
    - If a mutation strategy has dominated recent selections, choose a different mutation strategy unless all alternatives are clearly poor.
    - If a crossover strategy has dominated recent selections, choose a different crossover strategy unless all alternatives are clearly poor.
    - If all tested pairs performed poorly, select a less-tested or untested pair with reasonable exploration or balanced potential.
    - Untested pairs are allowed if the tested alternatives are overused or stagnant.
    - During early iterations, untested pairs should be preferred over repeatedly selecting the historically best pair.
    - Do not enumerate all combinations.
    - Do not rank all combinations.
    - Do not return multiple pairs.
    - Do not provide explanations.
    - Do not return JSON.
    - Do not return markdown.
    - Return only a Python-style vector with exactly two integers.
    
    Diversity of selection:
    - Avoid excessive reuse of a single pair across the run.
    - When changing operators, prefer changing at least one component of the current active pair.
    - Prefer a different mutation_id from the current active pair when the current mutation has been overused.
    - Prefer a different crossover_id from the current active pair when the current crossover has been overused.
    - In early iterations, prioritize diversity of selected operators over exploiting the best historical pair.
    
    Output format:
    [mutation_id, crossover_id]
    
    Valid examples:
    [6, 3]
    [7, 3]
    [8, 4]
    
    Invalid examples:
    "mutation 6 crossover 3"
    {{"mutation": 6, "crossover": 3}}
    [6, 3, 1]
    [0, 2]
    [11, 1]
    [3, 6]
    
    Return only the selected vector.
    """
    return prompt